In [1]:
import argparse
import inflect
import os
import pathlib
import utils
import kenlm
import csv

import numpy as np
import pandas as pd

from collections import defaultdict
from tqdm import tqdm
from functools import reduce
# from unigramlm import UnigramLM
from transformers import AutoTokenizer

/home/km55359/.conda/envs/kmisra/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/home/km55359/.conda/envs/kmisra/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
class UnigramLM:
    def __init__(self, counts_path):
        self.counts_path = counts_path
        self.model_name = counts_path.split("/")[-1].split(".")[0]
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                f"kanishka/smolm-autoreg-bpe-{self.model_name}-1e-3"
            )
        except:
            self.tokenizer = AutoTokenizer.from_pretrained(
                f"kanishka/smolm-autoreg-bpe-{self.model_name}-1e-4"
            )

    def load_counts(self):
        self.counts = {}
        with open(self.counts_path, "r") as f:
            reader = csv.DictReader(f)
            for line in reader:
                self.counts[line["word"]] = int(line["count"])
        self.total_counts = sum(self.counts.values()) + len(self.counts.keys())

    def sentence_log_prob(self, sentence, token_wise=False, normalize = False):
        words = self.tokenizer.tokenize(sentence)
        probs = []
        for word in words:
            if word in self.counts:
                probs.append((self.counts[word] + 1) / self.total_counts)
            else:
                probs.append(1 / self.total_counts)

        if token_wise:
            return [np.log(prob) for prob in probs]
        else:
            if normalize:
                return np.mean([np.log(prob) for prob in probs])
            else:
                print([np.log(prob) for prob in probs])
                return np.sum([np.log(prob) for prob in probs])

In [3]:
def compose(*functions):
    """compose functions"""
    return reduce(lambda f, g: lambda x: f(g(x)), functions, lambda x: x)


inflector = inflect.engine()

In [4]:
ngram2dir = {
    1: "unigrams",
    2: "bigrams",
    3: "trigrams",
    4: "fourgrams",
}

os.environ["TOKENIZERS_PARALLELISM"] = "false"
model_name = "babylm.csv"

model_name = model_name.split("/")[-1].split(".")[0]

aann_dir = "../data/mahowald-aann/"
aann_dir = aann_dir.split("data")[-1].strip("/")

unigram = f"../models/unigrams/{model_name}.csv"

model = "../models/babylm.csv"

In [5]:
unigram_lm = UnigramLM(unigram)
unigram_lm.load_counts()

# ngram_lm = model.replace(".csv", ".txt.binary")
ngram_lm = "../models/fourgrams/babylm.txt.binary"

lm = kenlm.Model(ngram_lm)
try:
    tokenizer = AutoTokenizer.from_pretrained(
        f"kanishka/smolm-autoreg-bpe-{model_name}-1e-3"
    )
except:
    tokenizer = AutoTokenizer.from_pretrained(
        f"kanishka/smolm-autoreg-bpe-{model_name}-3e-4"
    )
# pathlib.Path(args.results_dir).mkdir(parents=True, exist_ok=True)
# pathlib.Path(f"{args.results_dir}/{ngram2dir[args.ngram]}-alt").mkdir(
#     parents=True, exist_ok=True
# )

In [6]:
def n_gram_slor(prefix, continuation):
    unigram_logprob = unigram_lm.sentence_log_prob(" " + continuation, normalize=False)

    full = f"{prefix} {continuation}"
    full_tokenized = tokenizer.tokenize(" " + full)
    print(full_tokenized)
    full_length = len(full_tokenized)

    scores = list(lm.full_scores(" ".join(full_tokenized)))
    print(scores)
    # scores = [x * np.log(10) for x in scores]

    prefix_tokenized = tokenizer.tokenize(" " + prefix)
    # print(prefix_tokenized)
    prefix_len = len(prefix_tokenized)

    region = [p[0] for p in scores][prefix_len:-1]
    region = [x * np.log(10) for x in region]
    print(np.sum(region))

    print(full_length - prefix_len)

    # return (np.sum(region) / (full_length - prefix_len)) - unigram_logprob
    return (np.sum(region) - unigram_logprob) / (full_length - prefix_len)

In [7]:
np.log10(2) * np.log(10)

0.6931471805599454

In [9]:
n_gram_slor("The family spent", "a three days")

[-4.1494973938850706, -7.490331105212014, -8.37850108748256]
['Ġthe', 'Ġfamily', 'Ġspent', 'Ġa', 'Ġthree', 'Ġdays']
[(-1.3637523651123047, 2, False), (-2.850321054458618, 3, False), (-2.4238321781158447, 4, False), (-1.120948076248169, 4, False), (-3.750635862350464, 2, False), (-1.978846549987793, 3, False), (-1.8833470344543457, 3, False)]
-15.773699123611022
3


1.4148768209895415

In [47]:
unigram_lm.counts['Ġan'] + unigram_lm.counts['Ġaston']

277328

In [95]:
unigram_lm.sentence_log_prob(" an astonishing three days", normalize=False)

[-6.170281761872104, -11.443245078046749, -10.573090790756341, -7.490331105212014, -8.37850108748256]


-44.05544982336977

In [59]:
full = "The family spent an astonishing three days"
full_tokenized = tokenizer.tokenize(" " + full)
print(full_tokenized)
# list(lm.full_scores(" ".join(full_tokenized)))

['Ġthe', 'Ġfamily', 'Ġspent', 'Ġan', 'Ġaston', 'ishing', 'Ġthree', 'Ġdays']


In [64]:
full_tokenized = tokenizer.convert_ids_to_tokens(tokenizer("The family spent an astonishing three days").input_ids)

In [69]:
" ".join(full_tokenized)

'<s> Ġthe Ġfamily Ġspent Ġan Ġaston ishing Ġthree Ġdays'

In [83]:
# list(lm.full_scores(" ".join(full_tokenized)))
list(lm.full_scores("Ġthe Ġfamily Ġspent Ġan Ġaston ishing Ġthree Ġdays"))

[(-1.3637523651123047, 2, False),
 (-2.850321054458618, 3, False),
 (-2.4238321781158447, 4, False),
 (-2.625626564025879, 2, False),
 (-3.570128917694092, 2, False),
 (-0.08918390423059464, 3, False),
 (-2.5853962898254395, 4, False),
 (-2.0778491497039795, 2, False),
 (-1.6718428134918213, 3, False)]

In [93]:
list(lm.full_scores("Ġthe Ġfamily"))

[(-1.3637523651123047, 2, False),
 (-2.850321054458618, 3, False),
 (-2.295327663421631, 4, False)]